# Week-10, Part-2: Instance Segmentation Using YOLO v5 (Part-2)

### Welcome to the 10th Lab of 42028: Deep Learning and CNN!

In this part of the Lab/Tutorial session you will be exploring how to use the YOLO v5 for instance segmentation


So lets get started!


We will use the https://github.com/ultralytics/yolov5 repository for this tutorial

Reference: https://colab.research.google.com/github/ultralytics/yolov5/blob/master/segment/tutorial.ipynb

# Step 1: Clone github repository

#### **Custom Dataset Setup (We will use Toys dataset)**

In [ ]:
# Change path to you google drive location of the dataset as needed

## ENTER PATH TO DATASET ZIP FILE HERE ##
!unzip 'path/to/zip'

#### **Clone the Github Repo (Default location /content/)**

In [ ]:
# By Default Cloned under /content/
!git clone https://github.com/ultralytics/yolov5

Install the required libraries

In [ ]:
!pwd

In [ ]:
%cd yolov5
from yolov5 import utils
display = utils.notebook_init()  # checks Pytorch and GPU used
!pip install -r requirements.txt

#### **Quick Reference on YAML**

#### [**What is YAML?**](https://circleci.com/blog/what-is-yaml-a-beginner-s-guide/)

YAML --> **Y**et **A**nother **M**arkup **L**anguage

YAML is a digestible data serialization language often used to create configuration files with any programming language.

Designed for human interaction, YAML is a strict superset of JSON, another data serialization language. But because it’s a strict superset, it can do everything that JSON can and more. One major difference is that newlines and indentation actually mean something in YAML, as opposed to JSON, which uses brackets and braces.

Reference and Source: https://circleci.com/blog/what-is-yaml-a-beginner-s-guide/

# Step 2: Setup Training Configuration

In [ ]:
#@title Setup Training YAML File (Different than Object Detection)
number_of_classes = None #@param {type:"integer"}
with open('new_train_yaml', 'w+') as file:
    file.write(
        f"""
            # YOLOv5 🚀 by Ultralytics, AGPL-3.0 license

            # Parameters
            nc: {number_of_classes}  # number of classes
            depth_multiple: 0.33  # model depth multiple
            width_multiple: 0.5  # layer channel multiple
            anchors:
              - [10,13, 16,30, 33,23]  # P3/8
              - [30,61, 62,45, 59,119]  # P4/16
              - [116,90, 156,198, 373,326]  # P5/32

            # YOLOv5 v6.0 backbone
            backbone:
              # [from, number, module, args]
              [[-1, 1, Conv, [64, 6, 2, 2]],  # 0-P1/2
              [-1, 1, Conv, [128, 3, 2]],  # 1-P2/4
              [-1, 3, C3, [128]],
              [-1, 1, Conv, [256, 3, 2]],  # 3-P3/8
              [-1, 6, C3, [256]],
              [-1, 1, Conv, [512, 3, 2]],  # 5-P4/16
              [-1, 9, C3, [512]],
              [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
              [-1, 3, C3, [1024]],
              [-1, 1, SPPF, [1024, 5]],  # 9
              ]

            # YOLOv5 v6.0 head
            head:
              [[-1, 1, Conv, [512, 1, 1]],
              [-1, 1, nn.Upsample, [None, 2, 'nearest']],
              [[-1, 6], 1, Concat, [1]],  # cat backbone P4
              [-1, 3, C3, [512, False]],  # 13

              [-1, 1, Conv, [256, 1, 1]],
              [-1, 1, nn.Upsample, [None, 2, 'nearest']],
              [[-1, 4], 1, Concat, [1]],  # cat backbone P3
              [-1, 3, C3, [256, False]],  # 17 (P3/8-small)

              [-1, 1, Conv, [256, 3, 2]],
              [[-1, 14], 1, Concat, [1]],  # cat head P4
              [-1, 3, C3, [512, False]],  # 20 (P4/16-medium)

              [-1, 1, Conv, [512, 3, 2]],
              [[-1, 10], 1, Concat, [1]],  # cat head P5
              [-1, 3, C3, [1024, False]],  # 23 (P5/32-large)

              [[17, 20, 23], 1, Segment, [nc, anchors, 32, 256]],  # Detect(P3, P4, P5)
              ]
        """)

# Step 3: Setup Dataset paths

In [ ]:
#@title Setup Dataset Configuration (Data.yaml)
train_data_dir = "" #@param {type:"string"}
val_data_dir = "" #@param {type:"string"}
class_names = None #@param {type:"raw"}
with open('new_data_yaml', 'w+') as file:
    file.write(
        f"""
        train: {train_data_dir}
        val: {val_data_dir}

        nc: {number_of_classes}
        names: {class_names}
        """
    )

In [ ]:
# ---- OPTIONAL --------------
#@title Select YOLOv5 🚀 logger {run: 'auto'}
logger = 'TensorBoard' #@param ['TensorBoard', 'Comet', 'ClearML']

if logger == 'TensorBoard':
  %reload_ext tensorboard
  %tensorboard --logdir /content/yolov5/runs/train-seg/
elif logger == 'Comet':
  %pip install -q comet_ml
  import comet_ml; comet_ml.init()
elif logger == 'ClearML':
  import clearml; clearml.browser_login()

# ----------------------------

# Step 4: Start Training

**Configuation to try:**

*   **Image Size**: 416
*   **Batch Size**: 16
*   **Epochs**: 200
*   **Data Source details** : new_data_yaml (Created earlier)
*   **Training details** : new_train_yaml (Created earlier)

Example:
```shell
!python /content/yolov5/segment/train.py --img image_size --batch batch_size --epochs epochs --data /path/to/data/yml --cfg /path/to/train/yml
```

In [ ]:
## ENTER CODE TO START TRAINING ##

# Step 5: Test the trained model on sample images

### **Validation**

**Configuation to try:**

*   **Image Size**: 416
*   **Data Source details** : new_data_yaml (Created earlier)

Example:
```shell
!python /content/yolov5/segment/val.py --weights path/to/weithgs/best.pt --data path/to/data/yaml --img 416
```

In [ ]:
## ENTER CODE TO START EVALUATION ##

### **Inference/Testing**
'detect.py' is used to run YOLOv5 testing/inference on different types of inputs such as: image, video, webcam input, directory glob, Youtube, RTSP/RTMP/HTTP Stream.

**Usage Syntax:**

```shell
python segment/predict.py --source 0  # webcam
                             img.jpg  # image
                             vid.mp4  # video
                             screen  # screenshot
                             path/  # directory
                             'path/*.jpg'  # glob
                             'https://youtu.be/Zgi9g1ksQHc'  # YouTube
                             'rtsp://example.com/media.mp4'  # RTSP, RTMP, HTTP stream
                          --weights path/to/weights
                          --img 416
                          --conf 0.5
                          --save-txt
```
Example:
```shell
!python /content/yolov5/segment/predict.py --source '/path/to/img' --weights 'path/to/weights/best.pt' --img image_size --conf confidence_threshold --save-txt
```

In [ ]:
## ENTER CODE TO START INFERENCE ON THE IMAGES


# Step 6: Display result images

In [ ]:
import cv2
from matplotlib import pyplot as plt
from PIL import Image

# This is needed to display the images.
%matplotlib inline

In [ ]:
image = Image.open('/path/to/output/image')
plt.imshow(image)

# Step 7: Display performance analysis

Training results are automatically logged to Tensorboard and CSV as `results.csv`, which is plotted as `results.png` (below) after training completes. You can also plot any `results.csv` file manually:

In [ ]:
image = Image.open('/content/yolov5/runs/train-seg/exp/results.png') # Change 'exp' to the last in the train directory
plt.imshow(image)

In [ ]:
#[OPTIONAL] Alternate way of ploting the curves from CSV file
from utils.plots import plot_results
# Change the path to the last exp under train folder.
plot_results('/content/yolov5/runs/train-seg/exp/results.csv')  # plot 'results.csv' as 'results.png'

# Step 8: Optional (Video/Real-Time Inference)

### Try on a Road Traffic video with pre-trained weights

#### NOTE: **Download the output result video on local computer to visualize the results**

Download the Road Traffic video and upload to the SAGEMAKER storage, change the source path.

In [ ]:
!python ./segment/predict.py --source '../Road_Traffic.mp4'